# 數學計算 LLM Agent

最小可用的 tool-calling agent：

1. **Tool**：計算器 `calculate(expression)`。
2. **Tool Schema**：用 OpenAI 標準的 `tools` 參數告訴模型有哪些工具。
3. **Agent Loop**：模型要求呼叫工具 → 執行 → 把結果塞回對話 → 直到模型給出最終答案。

環境變數（`.env`）：`CILLM_API_KEY` + `CILLM_BASE_URL` 走課程 gateway；只設 `OPENAI_API_KEY` 則直連 OpenAI。

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

BASE_URL = os.getenv("CILLM_BASE_URL")  # 未設定時為 None，OpenAI SDK 會直連官方 API
API_KEY = os.getenv("CILLM_API_KEY") or os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("MODEL_NAME") or ("openai/gpt-oss-120b" if BASE_URL else "gpt-4o")

assert API_KEY, "請在 .env 設定 CILLM_API_KEY 或 OPENAI_API_KEY"

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(f"model = {MODEL}, endpoint = {BASE_URL or 'OpenAI API official endpoint'}")

## Tool：計算器

LLM 心算容易錯，所以計算交給程式。這裡最偷懶的寫法是直接 `eval(expression)`。

In [ ]:
def calculate(expression: str) -> float:
    return eval(expression)


# 測試計算器
print(calculate("312 * 0.87"))
print(calculate("(15 + 27) * 3 - 100 / 4"))

# 展示風險：下面這行是「合法運算式」，但 eval 會真的去讀檔案
# print(calculate("open('secret.txt', encoding='utf-8').read()"))

271.44
101.0
肚子很餓啊啊啊錒


## Tools Schema

`tools` 是給llm看的註冊表。

In [ ]:
tools = [
    {
        "type": "function",
        "name": "calculate",
        "description": "Calculate mathematical expressions.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A mathematical expression to evaluate",
                },
            },
            "required": ["expression"],
        },
    },
]

## Agent Loop（Responses API）

核心一樣是一個迴圈，但改用 `client.responses.create`：

- 回應的 `response.output` 是一串 item；型別為 `function_call` 的就是模型要呼叫的工具。
- 沒有 `function_call` → `response.output_text` 就是最終答案。
- 執行工具後，用 `{"type": "function_call_output", "call_id": ..., "output": ...}` 把結果接回 `input`，靠 `call_id` 對應。
- 記得先把模型這一輪的 `response.output` 原樣接回，模型才知道自己剛剛要求了什麼。
- `max_turns` 防止無限迴圈；工具出錯時把錯誤字串回給模型讓它自己重試。

In [ ]:
import json

SYSTEM_PROMPT = (
    "你是數學助理。遇到任何計算一律呼叫 calculate 工具，禁止心算。"
    "拿到工具結果後，用一句話回答並附上算式。"
)


def run_agent(question: str, max_turns: int = 10) -> str:
    # Responses API 用 input list，格式與 chat messages 類似
    input_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for _ in range(max_turns):
        response = client.responses.create(
            model=MODEL, input=input_messages, tools=tools
        )

        # 從 output 撈出模型要求的工具呼叫
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            return response.output_text

        # 把模型這一輪的 output（含 function_call）原樣接回對話
        input_messages += response.output

        for call in calls:
            args = json.loads(call.arguments)
            try:
                result = str(calculate(**args))
            except Exception as e:
                result = f"工具執行失敗：{e}"
            print(f"🔧 {call.name}({args}) → {result}")
            # Responses API 用 function_call_output 回傳結果，靠 call_id 對應
            input_messages.append(
                {
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": result,
                }
            )

    return "已達最大回合數，停止。"

## 試跑

In [ ]:
print(run_agent("312 個座位、載客率 87%，請問有多少旅客？"))

In [ ]:
print(run_agent("(15 + 27) * 3 - 100 / 4 等於多少？再把結果開平方。"))